# 패션 플랫폼 사용자 경험 분석 - Portfolio Summary

---

## 프로젝트 한 줄 요약

직접 수집한 설문 데이터로 패션 플랫폼 사용자의 **추천 의향, 구매 행동, 잔류 강도, UX 불만 요인**을 연결해 분석하고, 개선 우선순위를 정리했다.

> 이 노트북은 전체 분석의 포트폴리오용 요약본이다. 상세 전처리, 검정 과정, 세그먼트 정의는 각 세부 노트북과 `docs/` 문서에 분리했다.

---

_**데이터 출처** — 직접 수집한 설문 응답 (원본 269명 → 정제 266명 → 사용자 200명 → 구매자 191명). Google Forms 설문, 수집 기간 2026-04-25 ~ 2026-05-16, 편의표집. 원본 `raw_data/패션_플랫폼_설문_통합.csv`, 적재 MySQL `fashion_platform.survey`. 실제 앱 로그가 아닌 자기보고 단면 설문이며, 외부 매출·행동 데이터와의 검증은 후속 과제다._

---

## 프로젝트 배경

**왜 이 설문을 시작했나**
패션 앱은 무신사·에이블리·지그재그처럼 여러 개를 함께 쓰는 모습이 흔하고, 다른 앱으로 갈아타는 부담도 크지 않다. 그렇다면 사용자는 왜 특정 플랫폼에 남고, 적극적으로 추천하지도 않으면서 왜 계속 쓰는 걸까. 이 질문은 외부에서 행동 로그로 확인할 수 없어, 사용자에게 직접 물어보기로 했다 — 추천 의향(NPS), 실제 구매 행동, 그리고 불만을 한 설문에서 함께 본 이유다.

**어떻게 풀었나**
추천 의향(NPS)을 출발점으로 ① 실제 구매 행동과의 연결 → ② 설문 기반 RFM으로 우선 케어 대상 분리 → ③ 자유응답으로 UX 병목 식별 순서로, 의향·행동·텍스트를 한 흐름에서 교차 검토했다. (세부 질문은 아래 "1. 문제 정의")

**그래서, 무엇을 알게 됐나**
낮은 NPS(−32.0)가 곧 이탈을 뜻하지는 않았다. 다수는 불만이 있어도 익숙함·혜택 때문에 남는 **소극적 잔류층**에 가까웠다. 따라서 개선 우선순위는 단순 이탈 방어가 아니라 다음 순서로 정리된다.

| 우선순위 | 무엇을 | 근거 |
|---|---|---|
| 1 | 사이즈/핏·정보 신뢰성 개선 (세그먼트 공통 UX 병목) | 자유응답 92건 중 사이즈/핏 42.4%로 최다 |
| 2 | 객단가 신호 높은 재활성화 후보 우선 관찰 | 설문 기반 RFM에서 별도 분리 |
| 3 | 소극적 잔류층을 만족 사용자로 전환 | NPS는 낮으나 잔류 의향 93.5% |

> ⚠️ 자기보고 단면 설문 기반이므로 확정된 인과가 아니라 **방향성·우선순위 제안**이다. 실제 매출·행동 로그와의 검증은 후속 과제다.

---

## 1. 문제 정의

패션 플랫폼은 할인, 상품 다양성, 콘텐츠, 추천, 리뷰, 사이즈 정보 등 여러 요소가 동시에 작동하는 서비스다. 하지만 사용자가 플랫폼을 계속 쓰는 이유와 추천하지 않는 이유는 하나의 불만으로 설명하기 어렵다.

이 프로젝트는 다음 질문에서 출발했다.

| 핵심 질문 | 분석 관점 |
|---|---|
| 사용자는 왜 패션 플랫폼을 계속 쓰는가? | 재구매 이유, 계속 사용 의향, NPS |
| 추천 의향이 낮은 사용자는 실제 행동에서도 약한 신호를 보이는가? | NPS, 계속 사용 의향, 구매 빈도/최근 구매 시점 비교 |
| 우선 개선해야 할 사용자 경험 문제는 무엇인가? | 자유응답 텍스트, 불만족 경험, 세그먼트별 차이 |
| 어떤 사용자를 먼저 케어해야 하는가? | Survey-based RFM, 재활성화 후보군, 객단가 신호 높은 사용자 |

### 초기 가설

분석에 들어가며 세운 가설은 다음과 같다.

1. **추천 의향과 지속 이용 의향은 같은 방향으로 움직일 것이다.**
2. **추천 의향이 낮을수록 구매 빈도가 낮고 최근 구매 공백이 길 것이다.**
3. **반복되는 불편 경험은 낮은 추천 의향과 연결될 것이다.**

> 초기 단계에서는 NPS 수준이나 특정 불만의 크기를 전제하지 않았다. 세 가설이 분석 결과와 어떻게 맞물렸는지는 "4. 핵심 인사이트" 끝의 **분석 중 예상과 달랐던 지점**에 정리했다.

---

## 2. 데이터 개요

| 항목 | 내용 |
|---|---|
| 데이터 수집 방식 | Google Forms 직접 설문조사 (자기보고·단면) |
| 수집 기간 | 2026-04-25 ~ 2026-05-16 (편의표집) |
| 원본 데이터 파일 | `raw_data/패션_플랫폼_설문_통합.csv` (269행 × 20열, utf-8-sig) |
| 분석 DB | MySQL `fashion_platform.survey` (정제 후 266행) |
| 원본 응답 수 | 269명 |
| 분석 기준 응답 수 | 266명 (논리 모순 응답 3건 제외) |
| 패션 플랫폼 사용자 | 200명 (Q5 기준) |
| 최근 6개월 내 구매자 | 191명 (플랫폼 사용자 기준) |
| 주요 문항 | 플랫폼 사용 여부, 사용 플랫폼, 선택 요인, 구매 빈도, 최근 구매 시점, 평균 지출, 계속 사용 의향, NPS, 자유응답 |

### 데이터 해석 시 주의점

- 실제 앱 로그가 아니라 **설문 기반 자기보고 데이터**이므로 행동 해석에는 한계가 있다.
- 플랫폼, 선택 요인, 불만족 경험 등 일부 문항은 **다중응답**이므로 비율 비교 중심으로 해석했다.
- 표본은 20대 비중이 높아 전체 패션 플랫폼 시장으로 일반화하기보다, 20대 중심 표본의 경향으로 해석했다.


---

## 3. 분석 흐름

전체 분석은 분포 확인에서 시작해 추천 의향과 실제 행동을 연결하고, 후반에는 개선 대상 세그먼트와 UX 병목을 정리하는 흐름으로 구성했다.

| 단계 | 노트북 | 역할 |
|---|---|---|
| 1 | `01_cleaning.ipynb` | 원본 설문 정제, 변수 표준화, 모순 응답 제외 |
| 2 | `02_eda.ipynb` | 표본 특성, 플랫폼 이용 현황, 구매 행태 파악 |
| 3 | `03_nps.ipynb` | NPS 기반 추천 의향 세그먼트별 재구매 이유와 계속 사용 의향 분석 |
| 4 | `04_retention_and_behavior.ipynb` | 추천/사용 의향과 실제 구매 행동의 연결 검토 |
| 5 | `05_segmentation.ipynb` | Survey-based RFM으로 객단가 신호·재활성화 후보군 도출 |
| 6 | `06_channel.ipynb` | 인지 채널, 구매 영향 채널, 멀티호밍 패턴 분석 |
| 7 | `07_text_analysis.ipynb` | 자유응답 기반 UX 불만 요인 분석 |

---

## 4. 핵심 인사이트

### 인사이트 1. NPS는 낮지만, 즉시 이탈 의향자는 많지 않다 _(`03_nps`)_

패션 플랫폼 사용자의 NPS는 **-32.0**으로 추천 사용자보다 비추천 사용자가 많은 구조였다. 그러나 Detractor 중 명확하게 이탈 의향을 보인 사용자는 소수였다.

**해석**  
비추천 사용자가 곧바로 떠나지는 않는다. 상당수는 불만이 있어도 익숙함이나 혜택 때문에 사용을 유지하는 **소극적 잔류 사용자**에 가깝다.

**제품적 의미**  
단순 이탈 방어보다, 소극적 잔류자를 만족 사용자로 전환하는 과제를 먼저 볼 필요가 있다.

### 인사이트 2. 추천 의향은 사용 의향·구매 행동과 연결되지만, 단독 예측 변수는 아니다 _(`04_retention_and_behavior`)_

NPS는 계속 사용 의향과 연결됐고, 구매 빈도·최근 구매 시점과도 함께 봐야 했다. 다만 추천 점수 하나만으로 실제 행동을 강하게 설명하기는 어렵다.

**해석**  
추천 의향이 높은 사용자는 더 적극적으로 남는 경향을 보이지만, 패션 플랫폼 사용은 혜택·익숙함·멀티호밍 같은 요인이 함께 작동한다. NPS는 행동을 설명하는 출발점이지 단독 결론은 아니다.

**제품적 의미**  
NPS는 단독 KPI가 아니라 구매 빈도, 최근 구매 시점, 재구매 이유와 함께 봐야 한다.

### 인사이트 3. Survey-based RFM으로 객단가 신호 높은 재활성화 후보군을 분리할 수 있다 _(`05_segmentation`)_

구매 빈도, 최근 구매 시점, 평균 지출 금액을 조합해 사용자를 세분화하자, 재활성화가 필요한 사용자 안에서도 **객단가가 높은 사용자**가 확인됐다.

**해석**  
모든 재활성화 후보가 같은 우선순위를 갖지는 않는다. 최근 구매가 뜸해졌지만 과거 구매 금액이 높았던 사용자는 다시 활성화할 여지가 크다.

**제품적 의미**  
재방문 캠페인은 전체 휴면 사용자에게 똑같이 뿌리기보다, 객단가 신호 높은 재활성화 후보군을 우선 관찰하는 편이 더 효율적이다.

### 인사이트 4. 사이즈/핏이 핵심 불만이며, 추천/검색과 정보 신뢰성이 뒤를 잇는다 _(`07_text_analysis`)_

Q18 유효응답 92건을 전수 감사한 결과 사이즈/핏 42.4%, 추천/검색 26.1%, 정보/리뷰 22.8% 순으로 나타났다. UI/UX는 13.0%로, 기존 자동 분류에서 과대 집계됐던 결과를 원문 검토 후 수정했다.

**해석**  
온라인 패션 구매의 체형 적합성 불확실성이 가장 많이 언급됐고, 원하는 상품을 찾는 과정과 리뷰·사진 정보의 신뢰성도 반복적으로 나타났다. Detractor의 가격/혜택과 추천/검색 차이는 탐색적 신호로만 해석한다.

**제품적 의미**  
사이즈 표기 정규화, 검색·추천 다양성, 구조화된 리뷰 정보는 후속 검토 후보다. 구현 우선순위는 행동 로그, 효과와 비용을 추가로 확인한 뒤 정하는 편이 안전하다.

### 인사이트 5. 멀티호밍은 충성도 분산이 아니라 고관여 신호에 가깝다 _(`06_channel`)_

여러 패션 플랫폼을 함께 쓰는 사용자는 단일 플랫폼 사용자보다 NPS와 구매 빈도가 더 높은 경향을 보였다.

**해석**  
패션 플랫폼에서 여러 앱을 쓰는 행동이 곧 충성도가 낮다는 뜻은 아니다. 오히려 패션 관심도와 구매 관여도가 높은 사용자의 자연스러운 탐색 행동일 수 있다.

**제품적 의미**  
멀티호밍 사용자를 충성도 낮은 사용자로만 보지 말고, 고관여 사용자로 보고 차별화된 큐레이션과 리텐션 전략을 설계할 필요가 있다.

### 분석 중 예상과 달랐던 지점

초기 가설이 분석 결과와 어떻게 맞물렸는지 정리했다. 결과만 매끄럽게 남기기보다, 예상이 빗나간 지점과 그때 방향을 어떻게 틀었는지를 함께 둔다.

| 초기 가설 | 결과 | 근거 |
|---|---|---|
| 1. 추천 의향 = 지속 이용 의향 같은 방향 | **예상과 다르게 나타남** | NPS −32인데 지속 이용 의향 93.5% — 두 지표를 하나의 충성도 신호로 묶지 않기로 _(`03_nps`)_ |
| 2. 추천 낮을수록 구매 빈도↓·공백↑ | **방향성만 확인** | 방향은 맞으나 스피어만 약한 양의 관계 — NPS를 단독 지표로 쓰지 않기로 _(`04_retention_and_behavior`)_ |
| 3. 반복 불편 = 낮은 추천 의향 연결 | **집중 신호 없음** | Detractor 고유 불만 집중이 없어 사용자 전반의 공통 UX 병목으로 분석 방향 전환 _(`03_nps`·`07_text_analysis`)_ |

> **💡 해석**
> - 세 예상 중 둘이 빗나갔고, 그 지점이 오히려 이 분석의 핵심 발견으로 이어졌다.
> - 특히 3번은 저추천자 고유 문제를 찾기보다 공통 UX 병목 개선 과제로 방향을 튼 계기가 됐다.
> - 자기보고 단면 설문 기반이므로 확정이 아니라 방향성으로 해석한다.

---

## 5. 제품/마케팅 액션 제안

| # | 후속 검토 과제 | 근거 인사이트 |
|---|---|---|
| 1 | 사이즈/핏 정보 보강 — 실측·착용 후기·체형별 리뷰 노출 검토 | 인사이트 4 (사이즈/핏 42.4%, 선택형 불만 1위) |
| 2 | 실물-사진 격차 완화 — 실착 이미지·색감·상세 설명 보강 검토 | 인사이트 4 (정보/리뷰 22.8%, 선택형 불만 2위) |
| 3 | 탐색/추천 경험 점검 — 검색 필터·추천 노출 흐름 후속 점검 | 인사이트 4 (추천/검색 26.1%) |
| 4 | 잔류-추천 갭 모니터링 — Detractor 불만 원인·추천 저해 요인 추적 | 인사이트 1, 2 (잔류 의향 93.5% vs NPS -32) |
| 5 | 재구매·잔류 요인 점검 — 혜택·리워드 반응·구매 공백 재활성화 검토 | 인사이트 3, 5 (멀티호밍 81%·구매 공백률 15.7%) |

위 항목은 확정된 실행 우선순위가 아니라 분석 결과를 바탕으로 정리한 검토 과제다. 사용자의 상태와 문제 영역에 따라 다른 개입을 검증해야 한다.


---

## 6. 분석의 한계

- 이 데이터는 실제 앱 로그가 아니라 설문 기반 데이터이므로, 구매 행동은 자기보고에 의존한다.
- 표본은 20대 중심으로 구성돼 전체 연령대 일반화에는 한계가 있다.
- NPS Promoter 표본이 작아 세부 그룹 비교는 참고 수준으로 해석해야 한다.
- 다중응답 문항은 응답자 단위 독립성이 깨지므로 통계 검정보다 비율 비교와 정성 해석 중심으로 사용했다.
- Survey-based RFM은 실제 거래 로그 기반 RFM이 아니라 설문 문항을 점수화한 세그먼트이므로, 실제 매출 데이터와의 검증이 필요하다.

---

## 7. 다음 단계

후속 분석에서 설문 결과를 실제 행동 데이터와 연결하면 인사이트의 신뢰도를 높일 수 있다.

| 후속 과제 | 목적 |
|---|---|
| 실제 구매 로그와 연결 | 설문 기반 구매 빈도/최근 구매 시점 검증 |
| 앱 이벤트 로그 분석 | 탐색, 검색, 찜, 장바구니, 구매 전환 흐름 확인 |
| 사이즈/리뷰 개선 A/B 테스트 | UX 병목 개선 효과 측정 |
| 리텐션 캠페인 실험 | 객단가 신호 높은 재활성화 후보군의 캠페인 효과 검증 |
| 추가 표본 수집 | Promoter, Non-User 등 소표본 그룹 보강 |

---

## 8. 상세 분석으로 이동

| 보고 싶은 내용 | 파일 |
|---|---|
| 데이터 정제와 DB 적재 | `01_cleaning.ipynb` |
| 전체 분포와 EDA | `02_eda.ipynb` |
| NPS 세그먼트 분석 | `03_nps.ipynb` |
| 리텐션과 구매 행동 분석 | `04_retention_and_behavior.ipynb` |
| Survey-based RFM 세그먼트 | `05_segmentation.ipynb` |
| 채널과 멀티호밍 분석 | `06_channel.ipynb` |
| 자유응답 텍스트 분석 | `07_text_analysis.ipynb` |
| 변수 정의 | `../docs/variables.md` |
| 세그먼트 정의 | `../docs/segments.md` |
